# Visualize probabilistic forward-sim output vs. edepsim input

Loads `output/forward_simulation_output.h5` (probabilistic per-event distributions) and overlays the input edepsim segments. Segment coordinates are swapped (x ↔ z) to match the sim's internal frame (`TracksDataset(..., swap_xz=True)` in `optimize/simulate.py`).

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

BASE = 'PATH_TO_LARNDSIM_DIR'  # Replace with the actual path to the larndsim_position_prob_fwd directory

OUT_H5 = os.path.join(BASE, 'output/forward_simulation_output.h5')
IN_H5  = '/sdf/data/neutrino/cyifan/dunend_train_prod/prod_mod0_mpvmpr/production_884072/job_23771825_0000/output_23771825_0000-edepsim_lbl_trklen2cm_containment2cm_costheta0.966_range_0.05cm.h5'

EVENTS = [0, 2]  # global event IDs available in output/forward_simulation_output.h5

In [ ]:
def load_output_event(path, global_event_id):
    """Return dict of arrays for one event. Sets 'schema' based on which datasets are stored:
       - 'probabilistic': `ticks_prob` is present and has any non-zero entry.
       - 'hits'         : `ticks_prob` is missing (non-probabilistic write-path in simulate.py),
                          or present but entirely zero.
    Any dataset absent from the h5 group is simply absent from the returned dict — callers must
    check the schema (or use dict.get) before touching prob-only fields.
    """
    with h5py.File(path, 'r') as f:
        for bkey in f:
            g = f[bkey]
            ekey = f'event_{global_event_id}'
            if ekey not in g:
                continue
            ev = g[ekey]
            data = {k: ev[k][:] for k in ev.keys()}
            tp = data.get('ticks_prob')
            if tp is not None and np.any(tp != 0):
                data['schema'] = 'probabilistic'
            else:
                data['schema'] = 'hits'
            return data
    raise KeyError(f'event_{global_event_id} not found in {path}')

def load_input_segments(path, global_event_id):
    """Return segment records for one event, with x↔z swapped to match sim frame."""
    with h5py.File(path, 'r') as f:
        seg = f['segments']
        eids = seg['event_id'][:]
        mask = (eids == global_event_id)
        s = seg[mask]
    # swap x <-> z (matches TracksDataset(swap_xz=True))
    x_s, x_e = s['z_start'].copy(), s['z_end'].copy()
    z_s, z_e = s['x_start'].copy(), s['x_end'].copy()
    y_s, y_e = s['y_start'].copy(), s['y_end'].copy()
    return {
        'x_start': x_s, 'x_end': x_e,
        'y_start': y_s, 'y_end': y_e,
        'z_start': z_s, 'z_end': z_e,
        'dE': s['dE'],
        'pixel_plane': s['pixel_plane'],
    }

In [ ]:
def pixel_intensity(out):
    """Reduce (Npix, Nvalues, Nticks) distributions to per-pixel intensity.

    We threshold the ADC distribution at its per-event minimum (the digitized baseline
    from the electronics simulation) so only above-baseline entries count as 'hits'.
    """
    ad = out['adcs_distrib']
    baseline = ad.min()
    above = np.clip(ad - baseline, 0, None)  # (Npix, Nvalues, Nticks)
    per_pix = above.sum(axis=(1, 2))
    return per_pix, baseline

def tick_projection(out):
    """Marginalize over Nvalues to get intensity(pixel, tick)."""
    ad = out['adcs_distrib']
    baseline = ad.min()
    above = np.clip(ad - baseline, 0, None)
    return above.sum(axis=1), baseline  # (Npix, Nticks)

In [ ]:
# λ_h = Σ_t exp(max(ticks_prob[pix, h, t], min_log_prob))  — expected occurrence of hit slot h.
# Prints per pixel, per hit index, for every event. ticks_prob in the h5 is LOG-probability;
# we exp() with the same min_log_prob = -18.42 floor as fee_jax.get_average_hit_values.
# Skipped when the h5 file is a non-probabilistic write (no ticks_prob dataset).

MIN_LOG_PROB = -18.42

for ev_id in EVENTS:
    out = load_output_event(OUT_H5, ev_id)
    if out['schema'] != 'probabilistic':
        print(f'event {ev_id}: schema={out["schema"]} (no ticks_prob) — skipping λ table')
        continue
    tp = out['ticks_prob']  # (Npix, Nhits, Nticks) log-prob
    prob = np.exp(np.maximum(tp, MIN_LOG_PROB))
    lam = prob.sum(axis=2)  # (Npix, Nhits)
    Npix, Nhits = lam.shape

    print(f'=== event {ev_id}   shape={tp.shape} ===')
    header = 'pixel |   x     y   pl | ' + ' '.join(f'  h={h:<2d}' for h in range(Nhits)) + ' |   Σ_h'
    print(header)
    print('-' * len(header))
    for ip in range(Npix):
        row = ' '.join(f'{v:7.3g}' for v in lam[ip])
        print(f'{ip:5d} | {out["pix_x"][ip]:5.2f} {out["pix_y"][ip]:5.2f} {int(out["pixel_plane"][ip]):2d} '
              f'| {row} | {lam[ip].sum():7.3g}')
    print(f'Σ_pix per hit: ' + ' '.join(f'{v:7.3g}' for v in lam.sum(axis=0)))
    print()


In [ ]:
# Per-pixel non-zero probability distributions (masked by λ_h > LAMBDA_FLOOR).
# ticks_prob has shape (Npix, Nhits, Nticks) — the middle axis is the sequential hit index
# on the pixel; the stored values are log-probabilities.
# We compute λ_h = Σ_t exp(max(log_p, min_log_prob)) and only draw hit-index rows with λ_h > LAMBDA_FLOOR.
# Skipped when the h5 file is a non-probabilistic write (no ticks_prob dataset).
#
# For each event we pick the top-K pixels by Σ_h λ_h and show:
#   - LEFT : 2D heatmap of ticks_prob[pix] over (hit index, tick), floor-rows masked out (nan/white).
#   - RIGHT: (tick, ADC value) scatter for bins above PROB_FLOOR, colored by probability.

from matplotlib.colors import LogNorm

MIN_LOG_PROB = -18.42
LAMBDA_FLOOR = 1e-4   # skip hit-index rows whose Σ_t P (linear) falls below this
PROB_FLOOR = 1e-6     # bins with linear prob ≤ this are treated as "zero" for the scatter
TOP_K = 6             # number of pixels shown per event

def top_pixels_by_lambda(lam, k):
    order = np.argsort(-lam.sum(axis=1))
    return order[:k]

for ev_id in EVENTS:
    out = load_output_event(OUT_H5, ev_id)
    if out['schema'] != 'probabilistic':
        print(f'event {ev_id}: schema={out["schema"]} (no ticks_prob) — skipping per-pixel heatmaps')
        continue
    tp = out['ticks_prob']    # (Npix, Nhits, Nticks) — log-prob
    ad = out['adcs_distrib']  # (Npix, Nhits, Nticks)
    prob_lin = np.exp(np.maximum(tp, MIN_LOG_PROB))
    lam = prob_lin.sum(axis=2)                      # (Npix, Nhits)
    keep_row = lam > LAMBDA_FLOOR                   # (Npix, Nhits)

    # only pixels with at least one slot above the λ floor
    valid_pix = np.where(keep_row.any(axis=1))[0]
    if valid_pix.size == 0:
        print(f'event {ev_id}: no pixel has any hit slot with λ > {LAMBDA_FLOOR:g}')
        continue
    pxs = valid_pix[top_pixels_by_lambda(lam[valid_pix], k=min(TOP_K, valid_pix.size))]

    fig, axes = plt.subplots(len(pxs), 2, figsize=(12, 2.6*len(pxs)), squeeze=False)
    fig.suptitle(
        f'event {ev_id}: top {len(pxs)} pixels by Σ_h λ_h '
        f'(pixels with any λ_h > {LAMBDA_FLOOR:g}: {valid_pix.size} / {tp.shape[0]})',
        y=1.005,
    )

    for row, ip in enumerate(pxs):
        # LEFT: linear P(hit index, tick) for this pixel — mask rows with λ_h <= LAMBDA_FLOOR
        ax = axes[row, 0]
        pr = prob_lin[ip].copy()
        pr[~keep_row[ip]] = np.nan
        # Show it on a log color scale so the huge dynamic range renders
        vmin = max(pr[np.isfinite(pr) & (pr > 0)].min(initial=1e-12), 1e-12)
        vmax = pr[np.isfinite(pr)].max()
        cmap = plt.get_cmap('magma').copy()
        cmap.set_bad(color='white')
        im = ax.imshow(
            pr, aspect='auto', origin='lower',
            cmap=cmap, interpolation='nearest',
            norm=LogNorm(vmin=vmin, vmax=vmax),
        )
        # Annotate which rows survived the λ filter
        kept_rows = np.where(keep_row[ip])[0].tolist()
        ax.set_xlabel('tick index')
        ax.set_ylabel('hit index (0 = 1st trigger)')
        ax.set_title(
            f'pixel idx {ip}  (x={out["pix_x"][ip]:.2f}, y={out["pix_y"][ip]:.2f} cm, '
            f'plane={int(out["pixel_plane"][ip])}, kept rows={kept_rows})'
        )
        plt.colorbar(im, ax=ax, label='linear prob (λ-floor-masked)')

        # RIGHT: (tick, ADC) weighted by linear prob, only for surviving (h, t) bins above PROB_FLOOR
        ax = axes[row, 1]
        mask = (prob_lin[ip] > PROB_FLOOR) & keep_row[ip][:, None]
        if not mask.any():
            ax.text(0.5, 0.5, f'no bins above prob {PROB_FLOOR:g} in kept rows',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        _, t_idx = np.where(mask)
        adc_vals = ad[ip][mask]
        probs = prob_lin[ip][mask]
        sc = ax.scatter(
            t_idx, adc_vals, c=probs, s=22, cmap='magma',
            norm=LogNorm(vmin=max(probs.min(), 1e-30), vmax=probs.max()),
            edgecolors='none',
        )
        ax.set_xlabel('tick index')
        ax.set_ylabel('adcs_distrib value')
        ax.set_title(f'non-zero bins (prob > {PROB_FLOOR:g}, λ_h > {LAMBDA_FLOOR:g}): {int(mask.sum())}')
        plt.colorbar(sc, ax=ax, label='linear prob')

    plt.tight_layout()
    plt.show()


In [ ]:
# 3D event display (plotly). Handles both simulate.py output schemas:
#   - 'probabilistic': collapses each (pixel, hit-slot) with λ_h > LAMBDA_FLOOR to one dot placed
#     at argmax_t ticks_prob; color = adcs_distrib − baseline at that bin.
#   - 'hits' (non-probabilistic OR ticks_prob all zero): one dot per stored hit at (pix_x, pix_y, pix_z);
#     color = Q (charge, if present) else adc_clean else adc.
# Edepsim segments overlaid in red; TPC boundaries drawn from ref_params.tpc_borders.

import sys
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
src_path = os.path.join(BASE, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from larndsim.consts_jax import build_params_class, load_detector_properties
from larndsim.detsim_jax import get_hit_z

Params = build_params_class([])
ref_params = load_detector_properties(
    Params,
    os.path.join(BASE, 'src/larndsim/detector_properties/module0.yaml'),
    os.path.join(BASE, 'src/larndsim/pixel_layouts/multi_tile_layout-2.4.16_v4.yaml'),
)
tpc_borders = np.asarray(ref_params.tpc_borders)  # (n_tpc, 3, 2): [x,y,z][min,max]

MIN_LOG_PROB = -18.42
LAMBDA_FLOOR = 1e-4   # drop (pix, hit-slot) with Σ_t P below this (probabilistic schema only)


def _box_edges(bounds):
    """bounds: (3,2). Return (x, y, z) lists for the 12 edges of the axis-aligned box, None-separated."""
    (x0, x1), (y0, y1), (z0, z1) = bounds
    c = [(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),
         (x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    xs, ys, zs = [], [], []
    for a, b in edges:
        xs += [c[a][0], c[b][0], None]
        ys += [c[a][1], c[b][1], None]
        zs += [c[a][2], c[b][2], None]
    return xs, ys, zs


def collapsed_hits_prob(out, ref_params):
    """Probabilistic schema: one dot per (pixel, hit-slot) with λ_h > LAMBDA_FLOOR."""
    tp = out['ticks_prob']
    ad = out['adcs_distrib']
    prob_lin = np.exp(np.maximum(tp, MIN_LOG_PROB))
    lam = prob_lin.sum(axis=2)
    keep = lam > LAMBDA_FLOOR

    argmax_tick = tp.argmax(axis=2)
    Npix, Nhits = argmax_tick.shape
    pi = np.broadcast_to(np.arange(Npix)[:, None], (Npix, Nhits))
    hi = np.broadcast_to(np.arange(Nhits)[None, :], (Npix, Nhits))

    pix_idx = pi[keep]
    hit_idx = hi[keep]
    tick_at = argmax_tick[keep]
    charge = ad[pix_idx, hit_idx, tick_at] - ad.min()
    z = np.asarray(get_hit_z(
        ref_params,
        tick_at.astype(np.float32),
        out['pixel_plane'][pix_idx],
        fixed_v=True,
    ))
    hover = [f'pix {p} hit {h} λ={l:.3g}'
             for p, h, l in zip(pix_idx, hit_idx, lam[keep])]
    return dict(x=out['pix_x'][pix_idx], y=out['pix_y'][pix_idx], z=z,
                charge=charge, hover=hover, color_title='ADC - baseline')


def hits_from_nonprob(out):
    """Non-probabilistic schema: one dot per stored hit at (pix_x, pix_y, pix_z)."""
    charge = out.get('Q', out.get('adc_clean', out.get('adc')))
    label = 'Q' if 'Q' in out else ('adc_clean' if 'adc_clean' in out else 'adc')
    ticks = out.get('ticks')
    pixels = out.get('pixels')
    hover = [
        f'pix {int(p)} tick {int(t)} {label}={c:.3g}'
        for p, t, c in zip(
            pixels if pixels is not None else np.arange(len(out['pix_x'])),
            ticks if ticks is not None else np.zeros(len(out['pix_x']), dtype=int),
            charge,
        )
    ]
    return dict(x=out['pix_x'], y=out['pix_y'], z=out['pix_z'],
                charge=charge, hover=hover, color_title=label)


fig = make_subplots(
    rows=1, cols=len(EVENTS),
    specs=[[{'type': 'scene'}] * len(EVENTS)],
    subplot_titles=[f'event {ev}' for ev in EVENTS],
    horizontal_spacing=0.02,
)

for i, ev_id in enumerate(EVENTS):
    out = load_output_event(OUT_H5, ev_id)
    seg = load_input_segments(IN_H5, ev_id)
    if out['schema'] == 'probabilistic':
        hits = collapsed_hits_prob(out, ref_params)
        hits_name = f'collapsed hits (λ>{LAMBDA_FLOOR:g})'
    else:
        hits = hits_from_nonprob(out)
        hits_name = 'sim hits'

    scene = f'scene{i+1}' if i > 0 else 'scene'

    fig.add_trace(go.Scatter3d(
        x=hits['x'], y=hits['y'], z=hits['z'],
        mode='markers',
        marker=dict(
            size=3, color=hits['charge'], colorscale='Viridis', opacity=0.9,
            colorbar=dict(title=hits['color_title'], x=0.46 + 0.52*i, len=0.75),
        ),
        text=hits['hover'],
        hovertemplate='%{text}<br>x=%{x:.2f} y=%{y:.2f} z=%{z:.2f}<br>charge=%{marker.color:.2f}<extra></extra>',
        name=hits_name,
        showlegend=(i == 0),
    ), row=1, col=i+1)

    seg_x, seg_y, seg_z = [], [], []
    for j in range(len(seg['x_start'])):
        seg_x += [seg['x_start'][j], seg['x_end'][j], None]
        seg_y += [seg['y_start'][j], seg['y_end'][j], None]
        seg_z += [seg['z_start'][j], seg['z_end'][j], None]
    fig.add_trace(go.Scatter3d(
        x=seg_x, y=seg_y, z=seg_z,
        mode='lines',
        line=dict(color='red', width=3),
        name='edepsim segments',
        showlegend=(i == 0),
    ), row=1, col=i+1)

    for itpc in range(tpc_borders.shape[0]):
        bx, by, bz = _box_edges(tpc_borders[itpc])
        fig.add_trace(go.Scatter3d(
            x=bx, y=by, z=bz,
            mode='lines',
            line=dict(color='rgba(80,80,80,0.6)', width=2),
            name=f'TPC {itpc}',
            legendgroup='tpc_borders',
            showlegend=(i == 0 and itpc == 0),
        ), row=1, col=i+1)

    fig.update_layout({scene: dict(
        xaxis_title='x (sim frame) [cm]',
        yaxis_title='y [cm]',
        zaxis_title='z (drift) [cm]',
        aspectmode='data',
    )})
    print(f'event {ev_id}: schema={out["schema"]}, hits plotted={len(hits["x"])}')

fig.update_layout(height=700, width=900*len(EVENTS), margin=dict(l=0, r=0, b=0, t=40))
fig.show()
